# 04 — BERT Fine-Tuning

Fine-tune `bert-base-uncased` for binary classification.
BERT understands context much better than TF-IDF.

- Pre-trained on 3.3 billion words
- 3 epochs, lr=2e-5, batch_size=16, max_len=256
- Expected: 99%+ accuracy
- Training: ~20 min GPU, ~2 hours CPU

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
# AdamW moved from transformers to torch.optim in newer versions
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np

# Check if GPU is available (training is MUCH faster with GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
# If CPU --- training will take ~2 hours. GPU --- ~20 minutes

In [ ]:
# Load data
df = pd.read_csv('../datasets/processed/clean_data.csv')
texts = df['cleaned_text'].tolist()
labels = df['fraudulent'].tolist()

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# Load BERT tokenizer
# 'bert-base-uncased' = pre-trained on 3.3 billion words
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Create PyTorch Dataset class
class JobDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create DataLoaders
train_dataset = JobDataset(X_train, y_train, tokenizer)
test_dataset = JobDataset(X_test, y_test, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)
print('DataLoaders created!')

In [ ]:
# Load BERT model for binary classification (num_labels=2)
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2)
model = model.to(device)

# Optimizer and learning rate scheduler
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * 3  # 3 epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=total_steps)
print('Model and optimizer ready!')

In [ ]:
# Training loop — 3 epochs
EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_idx, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_batch = batch['label'].to(device)
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        if batch_idx % 50 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} — Loss: {loss.item():.4f}')
    
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{EPOCHS} — Avg Loss: {avg_loss:.4f}')

In [ ]:
# Evaluate
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy())

print(classification_report(all_labels, all_preds, target_names=['Real', 'Fake']))

In [ ]:
# Save the fine-tuned BERT model
import os
os.makedirs('../backend/models/bert_model', exist_ok=True)
model.save_pretrained('../backend/models/bert_model')
tokenizer.save_pretrained('../backend/models/bert_model')
print('BERT model saved to backend/models/bert_model/')